# Day 5: CSV Data Analysis with NumPy

Today we'll read a CSV of cereal data and perform statistical analysis using NumPy and Python lists/dicts.

The CSV contains columns: name, mfr, type, calories, protein, fat, sodium, fiber, carbo, sugars, potass, vitamins, shelf, weight, cups, rating.

## Stacks and Heaps

linsplace going from 0 to 4 — the memory address. Knowing integers are immutable: if we try to mutate an integer, we get a brand-new integer. This relates to what a **shallow copy** is.

## Reading the Cereal CSV

We can read in a CSV as follows:

In [ ]:
import numpy as np
import csv
with open('cereal.csv', 'r') as f:
    reader = csv.reader(f)
    data = list(reader)
data_array = np.array(data)
data_array

In [ ]:
len(data_array)

### Column names

The first row of the CSV is the header. Let's grab it and build a column-name-to-index lookup dict.

In [ ]:
header = data[0]
col_index = {name: i for i, name in enumerate(header)}
header
col_index

### Read individual columns into a NumPy array (as numbers)

We need to skip the header row and convert each value to a number using `float` or `int`.

In [ ]:
# Numeric columns (1-indexed in our tuple style, but 0-indexed in the CSV)
# name=0, mfr=1, type=2, calories=3, protein=4, fat=5, sodium=6,
# fiber=7, carbo=8, sugars=9, potass=10, vitamins=11, shelf=12, weight=13, cups=14, rating=15

names = [row[0] for row in data[1:]]

# Create a helper to read a numeric column
def numeric_column(col_name, cast=float):
    idx = col_index[col_name]
    return np.array([cast(row[idx]) for row in data[1:]])

calories = numeric_column('calories', int)
protein = numeric_column('protein', int)
fat = numeric_column('fat', int)
sodium = numeric_column('sodium', int)
fiber = numeric_column('fiber', float)
carbo = numeric_column('carbo', float)
sugars = numeric_column('sugars', float)
potass = numeric_column('potass', float)
vitamins = numeric_column('vitamins', int)
shelf = numeric_column('shelf', int)
weight = numeric_column('weight', float)
cups = numeric_column('cups', float)
rating = numeric_column('rating', float)

print(f'Loaded {len(names)} cereals with {len(header)} columns each.')
print('sugars[:5] =', sugars[:5])
print('type(sugars) =', type(sugars), 'dtype =', sugars.dtype)

## Sugar Analysis

Which cereal has the most and least sugar? What's the average amount of sugar? List all the cereals with below-average sugar.

In [ ]:
max_sugar_idx = np.argmax(sugars)
min_sugar_idx = np.argmin(sugars)

print(f'Most sugar: {names[max_sugar_idx]} with {sugars[max_sugar_idx]}g')
print(f'Least sugar: {names[min_sugar_idx]} with {sugars[min_sugar_idx]}g')

In [ ]:
avg_sugar = sugars.mean()
print(f'Average sugar: {avg_sugar:.2f}g')

In [ ]:
below_avg_sugar = [names[i] for i, s in enumerate(sugars) if s < avg_sugar]
print(f'Cereals with below-average sugar ({len(below_avg_sugar)} cereals):')
for c in below_avg_sugar:
    print(f'  {c}')

## Potassium: Percent of Aisle

Find out what percent aisle each cereal is in terms of potassium and print it out to the screen.

In [ ]:
# 'Percent aisle' = rank percentile: what percent of cereals have less potassium than this one?
# We compute a percentile rank for each cereal's potassium value.

sorted_potass = np.sort(potass)
n = len(potass)

percent_ranks = []
for i in range(n):
    # Count how many values are strictly less than potass[i]
    less_count = np.sum(potass < potass[i])
    # Percentile rank = (number less) / (total) * 100
    pr = (less_count / n) * 100
    percent_ranks.append(pr)

# Sort by percent rank for display
order = np.argsort(percent_ranks)
print('Cereal'.ljust(35), 'Potassium (mg)'.rjust(15), 'Percentile'.rjust(12))
print('-' * 62)
for idx in order:
    print(names[idx].ljust(35), f'{potass[idx]:>13.1f}', f'{percent_ranks[idx]:>11.1f}%')

## Fiber Analysis

Which cereals have no fiber? Which cereals have **no fiber and above-average sugar**?

Which cereals have **no fat and below-average sugar**?

In [ ]:
no_fiber = [names[i] for i, f in enumerate(fiber) if f == 0]
print(f'Cereals with no fiber ({len(no_fiber)}):')
for c in no_fiber:
    print(f'  {c}')

In [ ]:
# No fiber AND above-average sugar
avg_sugar_val = sugars.mean()
no_fiber_high_sugar = [names[i] for i in range(n) if fiber[i] == 0 and sugars[i] > avg_sugar_val]
print(f'Cereals with no fiber and above-average sugar ({len(no_fiber_high_sugar)}):')
for c in no_fiber_high_sugar:
    print(f'  {c}')

In [ ]:
# No fat AND below-average sugar
avg_sugar_val = sugars.mean()
no_fat_low_sugar = [names[i] for i in range(n) if fat[i] == 0 and sugars[i] < avg_sugar_val]
print(f'Cereals with no fat and below-average sugar ({len(no_fat_low_sugar)}):')
for c in no_fat_low_sugar:
    print(f'  {c}')

## Carbohydrates Statistics

What is the mean, standard deviation, and variance of carbohydrates? If you were carbo-loading, which cereal would you get?

In [ ]:
carb_mean = carbo.mean()
carb_std = carbo.std()
carb_var = carbo.var()

print(f'Mean carbohydrates:   {carb_mean:.2f}g')
print(f'Std dev carbohydrates: {carb_std:.2f}g')
print(f'Variance carbohydrates: {carb_var:.2f}')

# For carbo-loading, pick the cereal with the most carbs
max_carb_idx = np.argmax(carbo)
print(f'\nFor carbo-loading: {names[max_carb_idx]} with {carbo[max_carb_idx]}g of carbohydrates')

## Statistical Investigation

Create your own investigation. Use hypothesis testing or correlation analysis to answer a question.

Examples:
- "There is more variance in sugar than potassium."
- "Amount of protein is a good predictor of amount of vitamins."
- "Amount of sugar is an inverse predictor of amount of vitamins."

We'll investigate using correlation and a t-test.

In [ ]:
# Let's load scipy for a t-test
from scipy import stats

# Question: Is there a significant difference in sugar content between
# General Mills cereals and Kellogg cereals?
#
# The 'mfr' column uses single-letter codes:
#   G = General Mills
#   K = Kellogg
#   N = Nabisco
#   P = Post
#   Q = Quaker
#   R = Ralcorp
#   A = (American Homebird? — not present in this dataset)

mfr = [row[1] for row in data[1:]]

In [ ]:
gm_sugars = [sugars[i] for i in range(n) if mfr[i] == 'G']
kellogg_sugars = [sugars[i] for i in range(n) if mfr[i] == 'K']

print(f'General Mills cereals: {len(gm_sugars)} cereals, avg sugar = {np.mean(gm_sugars):.2f}g')
print(f'Kellogg cereals:       {len(kellogg_sugars)} cereals, avg sugar = {np.mean(kellogg_sugars):.2f}g')

t_stat, p_value = stats.ttest_ind(gm_sugars, kellogg_sugars)
print(f'\nt-statistic: {t_stat:.4f}')
print(f'p-value:     {p_value:.4f}')

if p_value < 0.05:
    print('Result: Significant difference in sugar content (p < 0.05)')
else:
    print('Result: No significant difference in sugar content (p >= 0.05)')

In [ ]:
# Hypothesis: 'There is more variance in sugar than potassium.'
sugar_var = sugars.var()
potass_var = potass.var()

print(f'Sugar variance:    {sugar_var:.2f}')
print(f'Potassium variance: {potass_var:.2f}')
print(f'Sugar has more variance: {sugar_var > potass_var}')

In [ ]:
# Hypothesis: 'Amount of sugar is an inverse predictor of amount of vitamins.'
# i.e., the more sugar, the fewer vitamins

corr_coef = np.corrcoef(sugars, vitamins)[0, 1]
print(f'Correlation between sugars and vitamins: {corr_coef:.4f}')

if corr_coef < 0:
    print('The negative correlation supports the hypothesis: sugar is an inverse predictor of vitamins.')
else:
    print('The correlation does not support the hypothesis.')

In [ ]:
# Hypothesis: 'Amount of protein is a good predictor of amount of vitamins.'

corr_protein_vit = np.corrcoef(protein, vitamins)[0, 1]
print(f'Correlation between protein and vitamins: {corr_protein_vit:.4f}')
print(f'Absolute correlation: {abs(corr_protein_vit):.4f}')

if abs(corr_protein_vit) > 0.7:
    print('Strong correlation -> protein is a good predictor of vitamins.')
elif abs(corr_protein_vit) > 0.4:
    print('Moderate correlation.')
else:
    print('Weak correlation -> protein is NOT a good predictor of vitamins.')

## Presidents Data Exercises

We also worked with presidents data to practice sorting, filtering, and list comprehensions.

Each tuple in `presidents_data` has the form: `(name, height, weight, is_smoker)`

In [ ]:
presidents_data = [ ("George Washington", 74, 175, True), ("John Adams", 67, 165, True), ("Thomas Jefferson", 74.5, 180, False), ("James Madison", 64, 100, False), ("James Monroe", 72, 190, False), ("John Quincy Adams", 67, 175, True), ("Andrew Jackson", 73, 140, True), ("Martin Van Buren", 66, 150, False), ("William Henry Harrison", 68, 160, False), ("John Tyler", 72, 160, True), ("James K. Polk", 68, 165, False), ("Zachary Taylor", 68, 170, False),  ("Millard Fillmore", 69, 185, False), ("Franklin Pierce", 70, 160, True), ("James Buchanan", 72, 180, True), ("Abraham Lincoln", 76, 180, False), ("Andrew Johnson", 70, 175, True), ("Ulysses S. Grant", 68, 165, True), ("Rutherford B. Hayes", 68.5, 170, False), ("James A. Garfield", 72, 185, True), ("Chester A. Arthur", 74, 225, True), ("Grover Cleveland", 71, 260, True), ("Benjamin Harrison", 66, 170, True), ("William McKinley", 67, 200, True), ("Theodore Roosevelt", 70, 200, False), ("William Howard Taft", 71.5, 340, False), ("Woodrow Wilson", 71, 175, False), ("Warren G. Harding", 72, 210, True), ("Calvin Coolidge", 70, 150, True), ("Herbert Hoover", 71.5, 185, True), ("Franklin D. Roosevelt", 74, 188, True), ("Harry S. Truman", 69, 175, False), ("Dwight D. Eisenhower", 70, 175, True), ("John F. Kennedy", 72, 175, True), ("Lyndon B. Johnson", 75.5, 210, True), ("Richard Nixon", 71.5, 175, True), ("Gerald Ford", 72, 195, True), ("Jimmy Carter", 69.5, 160, False), ("Ronald Reagan", 73, 185, True), ("George H.W. Bush", 74, 195, True), ("Bill Clinton", 74, 215, True), ("George W. Bush", 71.5, 190, True), ("Barack Obama", 73.5, 175, True), ("Donald Trump", 75, 240, False), ("Joe Biden", 72, 178, False), ]

In [ ]:
# Sort by height/weight ratio using a named function
def sumHDivW(item) -> int:
    return item[1] / item[2]

In [ ]:
presidents_data.sort(key=sumHDivW)

In [ ]:
# Sort by smoking status (False before True)
presidents_data.sort(key=lambda x: x[3])

In [ ]:
def checkSmoking(item):
    return item[3]

In [ ]:
presidentSmokers = filter(checkSmoking, presidents_data)
presidentSmokers = list(presidentSmokers)

In [ ]:
presidentSmokers

[('George Washington', 74, 175, True),
 ('John Adams', 67, 165, True),
 ('John Quincy Adams', 67, 175, True),
 ('Andrew Jackson', 73, 140, True),
 ('John Tyler', 72, 160, True),
 ('Franklin Pierce', 70, 160, True),
 ('James Buchanan', 72, 180, True),
 ('Andrew Johnson', 70, 175, True),
 ('Ulysses S. Grant', 68, 165, True),
 ('James A. Garfield', 72, 185, True),
 ('Chester A. Arthur', 74, 225, True),
 ('Grover Cleveland', 71, 260, True),
 ('Benjamin Harrison', 66, 170, True),
 ('William McKinley', 67, 200, True),
 ('Warren G. Harding', 72, 210, True),
 ('Calvin Coolidge', 70, 150, True),
 ('Herbert Hoover', 71.5, 185, True),
 ('Franklin D. Roosevelt', 74, 188, True),
 ('Dwight D. Eisenhower', 70, 175, True),
 ('John F. Kennedy', 72, 175, True),
 ('Lyndon B. Johnson', 75.5, 210, True),
 ('Richard Nixon', 71.5, 175, True),
 ('Gerald Ford', 72, 195, True),
 ('Ronald Reagan', 73, 185, True),
 ('George H.W. Bush', 74, 195, True),
 ('Bill Clinton', 74, 215, True),
 ('George W. Bush', 71.5, 19

In [ ]:
smokerHeights = [x[1] for x in presidentSmokers]

In [ ]:
sum(smokerHeights) / len(smokerHeights)

71.33928571428571

## List Exercises (6-14)

Use the following list as your starting point. (Tip: If you run the cell below, you can use the variables in cells beneath).

In [ ]:
letters = ['A', 'B', 'C', 'D', 'E', 'F', 'G']

#6. Use a function to get the length of the letters list. Display it.

In [ ]:
print(len(letters))

#7. Get the last item in the list, using a positive index.

In [ ]:
letters[6]  # or letters[len(letters) - 1]

#8. Get the last item in the list, using a negative index.

In [ ]:
letters[-1]

#9. Write a slice expression to return the letters D through F (inclusive).

In [ ]:
letters[3:6]

#10. Write a slice expression that returns the following list: ['A', 'D', 'G'].

In [ ]:
letters[::3]

#11. Based on what you learned in exercise 4, can you guess how to write a program to return a list of lowercase letters based on the letters list?

In [ ]:
[x.lower() for x in letters]

#12. For #11, did you iterate the list using a list comprehension or a for loop? Why?

#13. How could you return a copy of the list using a slice expression?

In [ ]:
letters[:]

#14. How could you return a list that looks like the list below?

```python
['G', 'F', 'E', 'D', 'C', 'B', 'A']
```

In [ ]:
letters[::-1]

#15. The range function returns a sequence — that is to say, it can be used as you would use a list inside a for-loop or list comprehension. For example:

```python
num_list = [x for x in range(1,6)]
print(num_list)
```

Output:
```python
[1, 2, 3, 4, 5]
```

Based on what you know about slice syntax, what would you expect the output of the following code to be?

In [ ]:
num_list = [x for x in range(1,6)]
num_list[::2]
# Expected output: [1, 3, 5] — every other element starting at index 0

#16. What do you predict will be printed by the code below?

```python
num_list = [x for x in range(1,6)]
num_list[0:2] = [99, 100]
print(num_list)
```

In [ ]:
num_list = [x for x in range(1,6)]
num_list[0:2] = [99, 100]
print(num_list)
# Prediction: [99, 100, 3, 4, 5]
# The slice 0:2 replaces elements at indices 0 and 1 with 99 and 100.

#17. Here's a slight variation on the code from #16. What do you expect will be printed by the following code?

```python
num_list = [x for x in range(1,6)]
other_list = num_list[:]
other_list[0:2] = [99, 100]
print(num_list)
```

In [ ]:
num_list = [x for x in range(1,6)]
other_list = num_list[:]  # shallow copy — independent list
other_list[0:2] = [99, 100]  # modifies the copy, not the original
print(num_list)
# Prediction: [1, 2, 3, 4, 5]
# other_list is a copy, so modifying it doesn't change num_list.